In [ ]:
# ============================================================
# STEP 1: Install Required Libraries
# ============================================================

!pip install -q pandas openpyxl matplotlib gradio langchain langchain-openai langchain-community faiss-cpu


# ============================================================
# STEP 2: Import Required Libraries
# ============================================================

import os
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


# ============================================================
# STEP 3: Global Variables
# ============================================================

uploaded_df = None
vectorstore = None
openai_api_key = None


# ============================================================
# STEP 4: Set OpenAI API Key
# ============================================================

def set_api_key(api_key):
    global openai_api_key

    if api_key is None or api_key.strip() == "":
        return "Please enter your OpenAI API key."

    openai_api_key = api_key
    os.environ["OPENAI_API_KEY"] = api_key

    return "OpenAI API key saved successfully."


# ============================================================
# STEP 5: Upload and Process Business File
# ============================================================

def load_business_file(file):
    global uploaded_df, vectorstore, openai_api_key

    if openai_api_key is None:
        return "Please enter and save your OpenAI API key first.", None

    if file is None:
        return "Please upload a CSV or Excel file.", None

    try:
        file_path = file.name

        if file_path.endswith(".csv"):
            uploaded_df = pd.read_csv(file_path)

        elif file_path.endswith(".xlsx") or file_path.endswith(".xls"):
            uploaded_df = pd.read_excel(file_path)

        else:
            return "Invalid file type. Please upload only CSV or Excel file.", None

        # Convert uploaded dataframe into text
        data_text = uploaded_df.to_string()

        # Split large data into smaller chunks
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=100
        )

        chunks = splitter.split_text(data_text)

        documents = [Document(page_content=chunk) for chunk in chunks]

        # Create embeddings
        embeddings = OpenAIEmbeddings()

        # Store chunks inside FAISS vector database
        vectorstore = FAISS.from_documents(documents, embeddings)

        preview = uploaded_df.head()

        message = f"""
File uploaded and processed successfully.

Total Rows: {uploaded_df.shape[0]}
Total Columns: {uploaded_df.shape[1]}

Columns Available:
{list(uploaded_df.columns)}
"""

        return message, preview

    except Exception as e:
        return f"Error while processing file: {str(e)}", None


# ============================================================
# STEP 6: Generate Dataset Summary
# ============================================================

def generate_data_summary():
    global uploaded_df

    if uploaded_df is None:
        return "Please upload a file first."

    try:
        summary = ""

        summary += "DATASET BASIC INFORMATION\n"
        summary += "=========================\n"
        summary += f"Total Rows: {uploaded_df.shape[0]}\n"
        summary += f"Total Columns: {uploaded_df.shape[1]}\n\n"

        summary += "COLUMN NAMES\n"
        summary += "============\n"
        summary += str(list(uploaded_df.columns)) + "\n\n"

        summary += "DATA TYPES\n"
        summary += "==========\n"
        summary += str(uploaded_df.dtypes) + "\n\n"

        summary += "MISSING VALUES\n"
        summary += "==============\n"
        summary += str(uploaded_df.isnull().sum()) + "\n\n"

        summary += "NUMERICAL SUMMARY\n"
        summary += "=================\n"
        summary += str(uploaded_df.describe(include="all")) + "\n\n"

        return summary

    except Exception as e:
        return f"Error while generating summary: {str(e)}"


# ============================================================
# STEP 7: Generate Chart
# ============================================================

def create_chart(column_name):
    global uploaded_df

    if uploaded_df is None:
        return None, "Please upload a file first."

    if column_name is None or column_name.strip() == "":
        return None, "Please enter a column name."

    column_name = column_name.strip()

    if column_name not in uploaded_df.columns:
        return None, f"Column '{column_name}' not found. Please enter a valid column name."

    try:
        plt.figure(figsize=(9, 5))

        # Numeric column chart
        if pd.api.types.is_numeric_dtype(uploaded_df[column_name]):
            uploaded_df[column_name].dropna().plot(kind="hist")
            plt.title(f"Distribution of {column_name}")
            plt.xlabel(column_name)
            plt.ylabel("Frequency")

        # Text/category column chart
        else:
            uploaded_df[column_name].value_counts().head(10).plot(kind="bar")
            plt.title(f"Top 10 Values in {column_name}")
            plt.xlabel(column_name)
            plt.ylabel("Count")

        chart_path = "insightforge_chart.png"

        plt.tight_layout()
        plt.savefig(chart_path)
        plt.close()

        return chart_path, "Chart generated successfully."

    except Exception as e:
        return None, f"Error while creating chart: {str(e)}"


# ============================================================
# STEP 8: Ask AI Question Using RAG
# ============================================================

def ask_ai_question(question):
    global uploaded_df, vectorstore, openai_api_key

    if openai_api_key is None:
        return "Please enter and save your OpenAI API key first."

    if uploaded_df is None or vectorstore is None:
        return "Please upload and process a business data file first."

    if question is None or question.strip() == "":
        return "Please enter a business question."

    try:
        # Retrieve relevant chunks from uploaded data
        retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
        relevant_docs = retriever.invoke(question)

        context = "\n\n".join([doc.page_content for doc in relevant_docs])

        # Create LLM
        llm = ChatOpenAI(
            model="gpt-4o-mini",
            temperature=0
        )

        # Prompt
        prompt = ChatPromptTemplate.from_template("""
You are InsightForge, an AI Business Intelligence Assistant.

Your job is to analyze the uploaded business data and answer the user's question.

Use only the given business data context.
Do not guess anything outside the provided data.

Explain the answer in simple business language.

Business Data Context:
{context}

User Question:
{question}

Provide the answer in this format:

1. Direct Answer
2. Key Insight
3. Reason
4. Business Recommendation
""")

        chain = prompt | llm | StrOutputParser()

        answer = chain.invoke({
            "context": context,
            "question": question
        })

        return answer

    except Exception as e:
        return f"Error while answering question: {str(e)}"


# ============================================================
# STEP 9: Generate Automatic Business Insights
# ============================================================

def generate_business_insights():
    global uploaded_df, openai_api_key

    if openai_api_key is None:
        return "Please enter and save your OpenAI API key first."

    if uploaded_df is None:
        return "Please upload a file first."

    try:
        # Use first 50 rows for simple AI analysis
        sample_data = uploaded_df.head(50).to_string()

        llm = ChatOpenAI(
            model="gpt-4o-mini",
            temperature=0
        )

        prompt = ChatPromptTemplate.from_template("""
You are a Business Intelligence Analyst.

Analyze the following business dataset sample.

Dataset Sample:
{sample_data}

Provide the output in this format:

1. Business Summary
2. Key Trends
3. Possible Problems
4. Business Opportunities
5. Recommendations
6. Suggested Charts

Use simple and clear language.
""")

        chain = prompt | llm | StrOutputParser()

        response = chain.invoke({
            "sample_data": sample_data
        })

        return response

    except Exception as e:
        return f"Error while generating business insights: {str(e)}"


# ============================================================
# STEP 10: Create Gradio App
# ============================================================

with gr.Blocks(title="InsightForge - AI Business Intelligence Assistant") as app:

    gr.Markdown("""
    # InsightForge - AI Business Intelligence Assistant

    Upload your business data and get AI-powered summaries, insights, recommendations, and charts.

    This project uses:
    - LangChain
    - RAG
    - OpenAI LLM
    - FAISS Vector Database
    - Gradio UI
    """)

    # --------------------------------------------------------
    # Tab 1: API Key
    # --------------------------------------------------------

    with gr.Tab("1. API Key"):
        gr.Markdown("### Enter your OpenAI API key")

        api_key_input = gr.Textbox(
            label="OpenAI API Key",
            type="password",
            placeholder="Paste your OpenAI API key here"
        )

        api_key_button = gr.Button("Save API Key")

        api_key_status = gr.Textbox(
            label="API Key Status"
        )

        api_key_button.click(
            fn=set_api_key,
            inputs=api_key_input,
            outputs=api_key_status
        )

    # --------------------------------------------------------
    # Tab 2: Upload Data
    # --------------------------------------------------------

    with gr.Tab("2. Upload Data"):
        gr.Markdown("### Upload your CSV or Excel business data file")

        file_input = gr.File(
            label="Upload CSV or Excel File"
        )

        upload_button = gr.Button(
            "Upload and Process Data"
        )

        upload_message = gr.Textbox(
            label="Upload Status",
            lines=8
        )

        data_preview = gr.Dataframe(
            label="Data Preview"
        )

        upload_button.click(
            fn=load_business_file,
            inputs=file_input,
            outputs=[upload_message, data_preview]
        )

    # --------------------------------------------------------
    # Tab 3: Data Summary
    # --------------------------------------------------------

    with gr.Tab("3. Data Summary"):
        gr.Markdown("### Generate basic summary of uploaded data")

        summary_button = gr.Button(
            "Generate Data Summary"
        )

        summary_output = gr.Textbox(
            label="Dataset Summary",
            lines=22
        )

        summary_button.click(
            fn=generate_data_summary,
            inputs=[],
            outputs=summary_output
        )

    # --------------------------------------------------------
    # Tab 4: Ask AI
    # --------------------------------------------------------

    with gr.Tab("4. Ask AI"):
        gr.Markdown("### Ask business questions about your uploaded data")

        question_input = gr.Textbox(
            label="Ask a Business Question",
            placeholder="Example: Which product has the highest sales?"
        )

        ask_button = gr.Button(
            "Ask InsightForge"
        )

        ai_answer = gr.Textbox(
            label="AI Answer",
            lines=18
        )

        ask_button.click(
            fn=ask_ai_question,
            inputs=question_input,
            outputs=ai_answer
        )

    # --------------------------------------------------------
    # Tab 5: Generate Chart
    # --------------------------------------------------------

    with gr.Tab("5. Generate Chart"):
        gr.Markdown("### Enter a column name to generate a chart")

        column_input = gr.Textbox(
            label="Enter Column Name",
            placeholder="Example: Sales, Region, Product, Category"
        )

        chart_button = gr.Button(
            "Create Chart"
        )

        chart_output = gr.Image(
            label="Generated Chart"
        )

        chart_message = gr.Textbox(
            label="Chart Status"
        )

        chart_button.click(
            fn=create_chart,
            inputs=column_input,
            outputs=[chart_output, chart_message]
        )

    # --------------------------------------------------------
    # Tab 6: Business Insights
    # --------------------------------------------------------

    with gr.Tab("6. Business Insights"):
        gr.Markdown("### Generate automatic AI business insights")

        insight_button = gr.Button(
            "Generate Business Insights"
        )

        insight_output = gr.Textbox(
            label="Business Insights",
            lines=24
        )

        insight_button.click(
            fn=generate_business_insights,
            inputs=[],
            outputs=insight_output
        )


# ============================================================
# STEP 11: Launch Application
# ============================================================

app.launch(share=True)